In [ ]:
from sentence_transformers import SentenceTransformer, evaluation, LoggingHandler
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
import logging
from datasets import load_dataset, load_from_disk

In [ ]:
logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO, handlers=[LoggingHandler()])

In [ ]:
teacher_model_name = "sentence-transformers/stsb-roberta-base-v2"
teacher_model = SentenceTransformer(teacher_model_name)
model_1 = SentenceTransformer("distilled/model-distillation-2025-08-22_22-19-23 4L RoBERTa [0, 7, 10, 11]/final")
model_2 = SentenceTransformer("distilled/model-distillation-2025-08-24_00-28-26 6L RoBERTa [0, 1, 7, 9, 10, 11]/final")

In [ ]:
logging.info("Load the STSB dataset")
# Load the STSB dataset: https://huggingface.co/datasets/sentence-transformers/stsb
stsb_eval_dataset = load_dataset("sentence-transformers/stsb", split="validation")
logging.info(stsb_eval_dataset)

In [ ]:
# Create an STSB Dataset Correlation Evaluator
dev_evaluator_stsb = EmbeddingSimilarityEvaluator(
    sentences1=stsb_eval_dataset["sentence1"],
    sentences2=stsb_eval_dataset["sentence2"],
    scores=stsb_eval_dataset["score"],
    # main_similarity=SimilarityFunction.COSINE,
    similarity_fn_names=['cosine',"dot","euclidean","manhattan"],
    name="sts-dev",
)

In [ ]:
# Similarity Correlation Metrics Evaluator
logging.info("Teacher Performance:")
dev_evaluator_stsb(teacher_model)

logging.info("[0, 7, 10, 11] Performance:")
dev_evaluator_stsb(model_1)

logging.info("[0, 1, 7, 9, 10, 11] Performance:")
dev_evaluator_stsb(model_2)

In [ ]:
eval_dataset = load_from_disk("datasets/distillation_evaluation_dataset_stsb-roberta/")
eval_sentences = eval_dataset["sentence"]
dev_evaluator_mse = evaluation.MSEEvaluator(eval_sentences, eval_sentences, teacher_model=teacher_model)

In [ ]:
# MSE between teacher model and student model
logging.info("[0, 7, 10, 11]:")
dev_evaluator_mse(model_1)

logging.info("[0, 1, 7, 9, 10, 11]:")
dev_evaluator_mse(model_2)

In [ ]:
# Trainable parameters
print(f"Number of parameters in the Teacher model: {sum(p.numel() for p in teacher_model.parameters() if p.requires_grad)}")
print(f"Number of parameters in the Student model: {sum(p.numel() for p in model_1.parameters() if p.requires_grad)}")
print(f"Number of parameters in the Student model: {sum(p.numel() for p in model_2.parameters() if p.requires_grad)}")